In [4]:
# =========================
# 01 FEATURE ENGINEERING
# Serie A Match Prediction
# =========================

import pandas as pd
import numpy as np

#from google.colab import files
from google.colab import drive
drive.mount('/content/drive')
# =========================
# LOAD DATASET
# =========================

#uploaded = files.upload()
PROJECT_PATH = "/content/drive/MyDrive/Serie_A_Forecast"
#df = pd.read_csv("I1_25_26.csv")

df = pd.read_csv(
    f"{PROJECT_PATH}/data/raw/I1_25_26.csv"
)
# =========================
# SELECT USEFUL COLUMNS
# =========================

columns_to_keep = [
    "Date",
    "Time",
    "HomeTeam",
    "AwayTeam",
    "FTHG",
    "FTAG",
    "FTR",
    "HTHG",
    "HTAG",
    "HTR",
    "HS",
    "AS",
    "HST",
    "AST",
    "HF",
    "AF",
    "HC",
    "AC",
    "HY",
    "AY",
    "HR",
    "AR"
]

df = df[columns_to_keep].copy()

# =========================
# CLEAN DATES
# =========================

df["Date"] = pd.to_datetime(
    df["Date"],
    dayfirst=True
)

df = df.sort_values(
    by="Date"
).reset_index(drop=True)

# =========================
# CREATE POINTS
# =========================

def home_points(result):

    if result == "H":
        return 3

    elif result == "D":
        return 1

    return 0


def away_points(result):

    if result == "A":
        return 3

    elif result == "D":
        return 1

    return 0


df["HomePoints"] = df["FTR"].apply(home_points)

df["AwayPoints"] = df["FTR"].apply(away_points)

# =========================
# FEATURE FUNCTIONS
# =========================

def avg_goals_scored(team, matches):

    goals = []

    for _, row in matches.iterrows():

        if row["HomeTeam"] == team:
            goals.append(row["FTHG"])

        else:
            goals.append(row["FTAG"])

    return np.mean(goals) if goals else np.nan


# =========================

def avg_goals_conceded(team, matches):

    goals = []

    for _, row in matches.iterrows():

        if row["HomeTeam"] == team:
            goals.append(row["FTAG"])

        else:
            goals.append(row["FTHG"])

    return np.mean(goals) if goals else np.nan


# =========================

def avg_points(team, matches):

    points = []

    for _, row in matches.iterrows():

        if row["HomeTeam"] == team:
            points.append(row["HomePoints"])

        else:
            points.append(row["AwayPoints"])

    return np.mean(points) if points else np.nan


# =========================

def avg_shots(team, matches):

    shots = []

    for _, row in matches.iterrows():

        if row["HomeTeam"] == team:
            shots.append(row["HS"])

        else:
            shots.append(row["AS"])

    return np.mean(shots) if shots else np.nan


# =========================

def avg_shots_target(team, matches):

    shots = []

    for _, row in matches.iterrows():

        if row["HomeTeam"] == team:
            shots.append(row["HST"])

        else:
            shots.append(row["AST"])

    return np.mean(shots) if shots else np.nan

# =========================
# BUILD FINAL DATASET
# =========================

rows = []

for i in range(len(df)):

    match = df.iloc[i]

    home_team = match["HomeTeam"]

    away_team = match["AwayTeam"]

    # =========================
    # PREVIOUS MATCHES
    # =========================

    previous_matches = df.iloc[:i]

    # =========================
    # LAST 5 GENERAL MATCHES
    # =========================

    home_prev = previous_matches[
        (previous_matches["HomeTeam"] == home_team) |
        (previous_matches["AwayTeam"] == home_team)
    ].tail(5)

    away_prev = previous_matches[
        (previous_matches["HomeTeam"] == away_team) |
        (previous_matches["AwayTeam"] == away_team)
    ].tail(5)

    # =========================
    # LAST 5 HOME / AWAY MATCHES
    # =========================

    home_home_prev = previous_matches[
        previous_matches["HomeTeam"] == home_team
    ].tail(5)

    away_away_prev = previous_matches[
        previous_matches["AwayTeam"] == away_team
    ].tail(5)

    # =========================
    # FINAL ROW
    # =========================

    row = {

        # BASIC INFO

        "Date": match["Date"],
        "HomeTeam": home_team,
        "AwayTeam": away_team,
        "FTR": match["FTR"],

        # GENERAL FORM

        "home_avg_goals_5":
            avg_goals_scored(home_team, home_prev),

        "away_avg_goals_5":
            avg_goals_scored(away_team, away_prev),

        "home_avg_goals_conceded_5":
            avg_goals_conceded(home_team, home_prev),

        "away_avg_goals_conceded_5":
            avg_goals_conceded(away_team, away_prev),

        "home_avg_points_5":
            avg_points(home_team, home_prev),

        "away_avg_points_5":
            avg_points(away_team, away_prev),

        # SHOTS

        "home_avg_shots_5":
            avg_shots(home_team, home_prev),

        "away_avg_shots_5":
            avg_shots(away_team, away_prev),

        "home_avg_shots_target_5":
            avg_shots_target(home_team, home_prev),

        "away_avg_shots_target_5":
            avg_shots_target(away_team, away_prev),

        # HOME / AWAY FORM

        "home_home_points_5":
            home_home_prev["HomePoints"].mean(),

        "away_away_points_5":
            away_away_prev["AwayPoints"].mean(),

        "home_home_goals_5":
            home_home_prev["FTHG"].mean(),

        "away_away_goals_5":
            away_away_prev["FTAG"].mean()
    }

    rows.append(row)

# =========================
# CREATE FINAL DATAFRAME
# =========================

df_final = pd.DataFrame(rows)

# =========================
# REMOVE NaN
# =========================

df_final = df_final.dropna().reset_index(drop=True)

# =========================
# FINAL CHECKS
# =========================

print(df_final.head())

print(df_final.shape)

print(df_final.isna().sum())

# =========================
# SAVE FINAL DATASET
# =========================


df_final.to_csv(
    f"{PROJECT_PATH}/data/processed/df_final.csv",
    index=False
)

#df_final.to_csv(
   # "df_final.csv",
   # index=False
#)

print("df_final.csv saved successfully.")

Mounted at /content/drive
        Date  HomeTeam AwayTeam FTR  home_avg_goals_5  away_avg_goals_5  \
0 2025-09-13  Cagliari    Parma   H               0.5               0.5   
1 2025-09-14      Roma   Torino   A               1.0               0.0   
2 2025-09-14  Atalanta    Lecce   H               1.0               0.0   
3 2025-09-14      Pisa  Udinese   A               0.5               1.5   
4 2025-09-14  Sassuolo    Lazio   H               1.0               2.0   

   home_avg_goals_conceded_5  away_avg_goals_conceded_5  home_avg_points_5  \
0                        1.0                        1.5                0.5   
1                        0.0                        2.5                3.0   
2                        1.0                        1.0                1.0   
3                        1.0                        1.0                0.5   
4                        2.5                        1.0                0.0   

   away_avg_points_5  home_avg_shots_5  away_avg_shots